<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_0_split_train_valid_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_0_split_train_valid_test


## Introducción y Resumen

Esta notebook tiene como objetivo dividir el dataset MNQ en conjuntos de entrenamiento, validación y prueba, asegurando una partición aleatoria, reproducible y estructuralmente consistente. Con ello se dejan listos los datos para el entrenamiento y evaluación de los modelos predictivos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente (mnq_technical_indicators y mnq_alpha_factors) y se muestra un resumen de la información del dataset MNQ.

1. Carga de datos

    Se importa el dataset procesado con features técnicos y alpha factors. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Análisis del dataset `mnq_model`

    Se revisa la estructura del dataset (filas, columnas, tipos de datos), se busca los valores NaNs y se verifica la distribución temporal de los registros.

3. Definición de parámetros de división

    En este punto se define la estrategia de partición del dataset: se toma un 70% de los días para entrenamiento, y el 30% restante se divide en partes iguales para validación y prueba. De esta manera, el modelo cuenta con suficientes datos para aprender, mientras que se reservan bloques temporales separados para ajustar parámetros y evaluar el rendimiento final sin fugas de información.

4. Selección aleatoria de días.

    Este punto busca garantizar que la partición de los datos sea representativa y no esté sesgada por la secuencia temporal. Al asignar los días de forma aleatoria —aunque de manera reproducible— se evita que los conjuntos queden condicionados por períodos específicos del mercado (por ejemplo, tendencias prolongadas o alta volatilidad en ciertos meses). Así, cada subconjunto refleja mejor la diversidad del dataset y se obtiene una evaluación más robusta del modelo.

5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`

    En este punto se crean los datasets mnq_train, mnq_valid y mnq_test, manteniendo homogeneidad en estructura (301 registros por día, de 09:30 a 14:30) y sin solapamiento entre conjuntos. Esto asegura consistencia en el entrenamiento, validación y prueba del modelo.

## 0. Configuración del Entorno


### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [2]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [3]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr


## 1. Carga de datos

### 1.1. Carga de dataset `mqn_to_model`




In [4]:
def load_data(data: str):

    data_path = f'{drive_path}/2_feature_engineering/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [5]:
mnq_model = load_data("to_model")

In [6]:
#mnq_model

### 1.2. Información de dataset MNQ_to_model


In [7]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [8]:
num_dias, promedio_por_fecha = info_dataset(mnq_model, 'mnq_model')

Información del dataset mnq_model:

	Cantidad de días: 1311
	Registros por día: 481
	Hora diaria de inicio 08:00
	Hora diaria de final 16:00
	Zona horaria: America/New_York



### 1.3. Carga de listado de features por ventana de tiempo

In [20]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_90 = features_dict["features_to_90"]


In [21]:
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


In [22]:
features_base = ['open', 'high', 'low', 'close', 'volume']
features_90 = features_base + features_to_90

## 2. Análisis del dataset `mnq_model`

### 2.0. Funciones

#### Función para contar NaN en dataset

In [11]:
def nan_count(df):
  # Contar NaN por día y por columna
  daily_nan_counts = df.groupby("date").apply(lambda x: x.isna().sum())

  # Construir DataFrame con los valores únicos
  daily_unique_nans = pd.DataFrame({
      "feature": daily_nan_counts.columns,
      "daily_nan_counts": [sorted(daily_nan_counts[col].unique()) for col in daily_nan_counts.columns]
  })
  return daily_unique_nans

#### Función para detectar saltos temporales (gaps)

In [12]:
def detectar_gaps(df: pd.DataFrame, gap_minutes: int = 1):
    """
    Verifica si existen saltos mayores al intervalo esperado (por defecto 1 minuto)
    entre registros consecutivos dentro de cada día, en un DataFrame con índice tipo DatetimeIndex.

    Omite el primer registro de cada día.

    Parámetros:
    - df: DataFrame con índice datetime.
    - gap_minutes: tamaño esperado del intervalo en minutos (por defecto 1).

    Retorna:
    - Lista de índices donde se detectaron diferencias mayores al intervalo esperado.
    """
    df = df.copy()
    df['time_diff'] = df.index.to_series().diff()

    base_time_diff = pd.Timedelta(minutes=gap_minutes)
    problem_indices = []

    for date, group in df.groupby(df.index.date):
        time_diff = group['time_diff'].iloc[1:]
        incorrect_indices = time_diff[time_diff != base_time_diff].index
        if len(incorrect_indices) > 0:
            problem_indices.append(incorrect_indices)

    if problem_indices:
        print(f"Se encontraron problemas en {len(problem_indices)} registros con diferencias irregulares.\n")

        # Conteo por fecha
        conteos = df.groupby(df.index.date).size()

        for i in range(len(problem_indices)):
            idx = problem_indices[i][0]
            diff = df.loc[idx, 'time_diff']
            date = idx.date()
            count = conteos[date]
            print(f'\t{idx} -> Diferencia: {diff} | # Registros: {count}')
    else:
        print("No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.")

    return problem_indices

### 2.1. Búsqueda de NaN en `mnq_model`:

Buscamos los valores NaN en todas la columnas del dataset:

In [13]:
mnq_model_nans = nan_count(mnq_model)
mnq_model_nans

,feature,daily_nan_counts
0,date,[0]
1,open,[0]
2,high,[0]
3,low,[0]
4,close,[0]
5,volume,[0]
6,target_return_30,[30]
7,target_return_60,[60]
8,target_return_90,[90]
9,bb_60,[59]


Lo que se observa es:

- OHLCV y date → [0] → nunca tienen valores faltantes.

- Targets (target_return_*) → [30], [60], [90] → todos los días tienen esos NaN al final, consistente con la ventana de predicción que corta datos futuros.

- Factores técnicos → muchos muestran valores únicos iguales al tamaño de la ventana usada en su cálculo:

  - bb_60 → [59] → se necesitan 60 valores para calcular, por eso hay 59 NaN iniciales cada día.
  - ire_60, roc_60, rev_mom_vol_z_60 → [60].
  - ire_90, rev_mom_z_90 → [90].
  - roc_20 → [20].
  - momentum_5 → [5].

Caso particular:

- rev_score_90 → [1, 2, 3] → parece que en algunos días puede generar hasta 3 NaN, pero no es fijo como los demás.

Las ventanas más grandes con valores NaN son la `ire_90` y `rev_mom_z_90` que necesitan 90 minutos de historial.

Vamos a filtrar el dataset `mnq_model` para eliminar todos los NaNs:

In [14]:
# Filtrar filas sin NaN en ninguna columna
mnq_model_clean = mnq_model.dropna(how="any")

Verificamos si efectivamente no hay más NaNs en el dataset `mnq_model_clean`

In [15]:
mnq_model_clean_nans = nan_count(mnq_model_clean)
mnq_model_clean_nans

,feature,daily_nan_counts
0,date,[0]
1,open,[0]
2,high,[0]
3,low,[0]
4,close,[0]
5,volume,[0]
6,target_return_30,[0]
7,target_return_60,[0]
8,target_return_90,[0]
9,bb_60,[0]


Se comprueba que no contamos con valores NaN. Ahora observemos la información del dataset:

In [16]:
num_dias, promedio_por_fecha = info_dataset(mnq_model_clean, 'mnq_model_clean')

Información del dataset mnq_model_clean:

	Cantidad de días: 1311
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



Como se observa en el resultado, el primer registro válido (sin valores NaN en ninguna columna) aparece a las 09:30 y la jornada finaliza a las 14:30.

A continuación, verificamos en el dataset filtrado si existen saltos en la secuencia temporal o si todos los registros se mantienen consecutivos.

### 2.2. Búsqueda de gaps en `mnq_model_clean`

In [17]:
detectar_gaps(mnq_model_clean)

No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.


[]

Contamos con el dataset limpio de NaNs y saltos temporales.

In [19]:
mnq_model=mnq_model_clean.copy()

### 2.3. Filtramos los features correspondientes

In [30]:
mnq_model.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'target_return_30',
       'target_return_60', 'target_return_90', 'bb_60', 'ire_60', 'ire_90',
       'momentum_5', 'price_ema30', 'rev_mom_vol_z_60', 'rev_mom_z_90',
       'rev_score_90', 'roc_20', 'roc_60'],
      dtype='object')

In [28]:
features_90

['open',
 'high',
 'low',
 'close',
 'volume',
 'ire_60',
 'rev_mom_z_90',
 'roc_60',
 'bb_60',
 'momentum_5',
 'roc_20',
 'rev_mom_vol_z_60']

In [36]:
#Agregamos el features 'date' al inicio
features_90 = ['date'] + [col for col in features_90 if col != 'date']+['target_return_90']

In [37]:
mnq_90 = mnq_model[features_90].copy()

In [42]:
n_dias_90, promedio_por_fecha_90 = info_dataset(mnq_90, 'mnq_90')

Información del dataset mnq_90:

	Cantidad de días: 1311
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



### 2.3. Guardamos el dataset limpio

In [40]:
#Guardamos el dataset
ruta_mnq_90 = f'{drive_path}/5_transformer_90_model/mnq_90.parquet'
mnq_90.to_parquet(ruta_mnq_90, index=True)

## 3. Definición de parámetros de división

Para dividir el dataset en subconjuntos, se utiliza la siguiente estrategia:

- 70% de los días se asignan al conjunto de entrenamiento (train).

- El 30% restante se reparte de manera equitativa entre los conjuntos de validación (valid) y prueba (test).

Esto garantiza que el modelo disponga de la mayor parte de los datos para aprender patrones, mientras que las particiones de validación y prueba permiten ajustar hiperparámetros y evaluar el rendimiento fuera de muestra.

De esta forma, se asegura un esquema de división temporalmente consistente, sin solapamiento entre conjuntos.

In [56]:
num_dias = n_dias_90

In [60]:
n_train = int(num_dias * 0.7)
print(f'Tamaño dataset train: {n_train} días')

n_val = int((num_dias-n_train)/2)
print(f'Tamaño dataset valid: {n_val} días')

n_test = int((num_dias-n_train)/2)
print(f'Tamaño dataset test: {n_test} días')

Tamaño dataset train: 917 días
Tamaño dataset valid: 197 días
Tamaño dataset test: 197 días


## 4. Selección aleatoria de días.

En este punto nos aseguramos que la división de los datos no dependa únicamente del orden cronológico de los días, sino que se realice una selección aleatoria controlada. Para ello:

  - Se extraen los días únicos presentes en el dataset.
  - Se fijan 10 semillas distintas para garantizar reproducibilidad en la mezcla.
  - Se aplica `np.random.shuffle` para reordenar los días de manera aleatoria.
  - Finalmente, se asignan los días a los conjuntos de entrenamiento, validación y prueba de acuerdo con los tamaños definidos previamente (917, 197 y 197 días respectivamente).

Este procedimiento nos permite que cada subconjunto mantenga independencia respecto a los demás, evitando sesgos por orden temporal y asegurando que los modelos se entrenen, validen y evalúen sobre muestras representativas del universo completo de días.

In [87]:
#Generamos 10 semillas aleatorias pero fijas:
# Generamos 10 semillas aleatorias reproducibles
np.random.seed(2025)
while len(seeds) != len(set(seeds)):
  seeds = np.random.randint(0, 100, size=10)

# Verificar unicidad
all_unique = len(seeds) == len(set(seeds))
print("¿Todas son únicas?:", all_unique)
print("Semillas generadas:", seeds)



¿Todas son únicas?: True
Semillas generadas: [10 78 97 40 23 84 16 64  3 25]


In [88]:
seeds # = [10 78 97 40 23 84 16 64  3 25]

array([10, 78, 97, 40, 23, 84, 16, 64,  3, 25])

In [91]:
particiones = []

for seed in seeds:
    unique_days = mnq_90['date'].unique().copy()

    np.random.seed(seed)
    np.random.shuffle(unique_days)

    train_days = unique_days[:n_train]
    val_days   = unique_days[n_train:n_train + n_val]
    test_days  = unique_days[n_train + n_val:n_train + n_val + n_test]

    particiones.append({
        "seed": seed,
        "train_days": train_days,
        "val_days": val_days,
        "test_days": test_days
    })

## 5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`

En este paso generamos los 10 grupos distintos de datasets finales para cada subconjunto: `mnq_train`, `mnq_valid` y `mnq_test`. La asignación se realiza filtrando los días correspondientes a cada semilla (seed), lo que asegura que no exista solapamiento entre ellos.

In [100]:
for part in particiones:
    seed = part["seed"]
    #print(f'\tSeed {seed}')

    train_days = part["train_days"]
    val_days   = part["val_days"]
    test_days  = part["test_days"]

    globals()[f"mnq_train_{seed}"] = mnq_90[mnq_90['date'].isin(train_days)].copy()
    globals()[f"mnq_valid_{seed}"] = mnq_90[mnq_90['date'].isin(val_days)].copy()
    globals()[f"mnq_test_{seed}"]  = mnq_90[mnq_90['date'].isin(test_days)].copy()


Para un análisis posterior usamos la función antes definida `info_dataset` para confirmar que todos los subconjuntos comparten las mismas características estructurales:

In [104]:
info_dataset(mnq_train_64, 'mnq_train_64')
info_dataset(mnq_train_23,  'mnq_train_23')
info_dataset(mnq_train_97, 'mnq_train_97')

Información del dataset mnq_train_64:

	Cantidad de días: 917
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_train_23:

	Cantidad de días: 917
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_train_97:

	Cantidad de días: 917
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



(917, np.float64(301.0))

Se verifica que todos los datasets poseen una base homogénea, lo que nos va a facilitar la comparación de resultados entre etapas de entrenamiento, ajuste y evaluación. Además, la consistencia en el número de registros por día nos garantiza que los modelos reciban siempre ventanas de información con la misma extensión temporal.

## 6. Guardamos los datasets generados


In [106]:
for part in particiones:
    seed = part["seed"]

    # Rutas para cada dataset
    ruta_mnq_train = f"{drive_path}/5_transformer_90_model/5_0_train_valid_test/mnq_train_{seed}.parquet"
    ruta_mnq_valid = f"{drive_path}/5_transformer_90_model/5_0_train_valid_test/mnq_valid_{seed}.parquet"
    ruta_mnq_test  = f"{drive_path}/5_transformer_90_model/5_0_train_valid_test/mnq_test_{seed}.parquet"

    # Guardado
    globals()[f"mnq_train_{seed}"].to_parquet(ruta_mnq_train, index=True)
    globals()[f"mnq_valid_{seed}"].to_parquet(ruta_mnq_valid, index=True)
    globals()[f"mnq_test_{seed}"].to_parquet(ruta_mnq_test, index=True)